# MedVision AI: train the ISIC 2019 model on Kaggle

This notebook prepares the data, trains, evaluates and exports the skin-lesion
classifier with the repository's own `medvision-ml` CLI, so what runs here is
what CI tests.

**Before you start** (in the notebook's right-hand panel, *Session options*):
1. *Accelerator* → **GPU T4 x2**.
2. *Internet* → **On** (needs a phone-verified Kaggle account).

**Run it in the background:** *Save Version* → *Save & Run All (Commit)* → *Save*.
The run continues on Kaggle's servers after you close the browser (up to 12 hours;
expect about 3–5). Follow it under *View Active Events*.

**Results** appear in the saved version's *Output* tab:
`skin-effb0-v2.0.0.zip` (the serving files), `model/` (the Keras model) and
`reports/` (leakage check and test-set evaluation).

It downloads about 10 GB (ISIC 2019, CC BY-NC 4.0). Any failing step stops the
run, so a broken step never reaches the export.

## 1. Helpers, GPU and disk check

In [ ]:
import os
import subprocess

# Kaggle keeps only /kaggle/working (up to 20 GB) as output; the dataset and the
# image cache live in temporary space.
OUTPUT = "/kaggle/working"
WORK = "/tmp/medvision"
os.makedirs(WORK, exist_ok=True)


def sh(command: str) -> None:
    """Run a shell command, stream its output, and stop the run if it fails."""
    process = subprocess.Popen(
        command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")


sh("nvidia-smi -L")  # fails here if no GPU accelerator is selected
sh(f"df -h {WORK} {OUTPUT} && free -g")

## 2. Get the code and install dependencies

The repository pins TensorFlow; the CUDA libraries for that exact version are added on top.

In [ ]:
sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
# Kaggle points Matplotlib at its notebook backend, which does not exist in
# the project's own environment; TensorFlow imports Matplotlib, so use the
# headless backend everywhere.
os.environ["MPLBACKEND"] = "agg"

REPO = f"{WORK}/MedVisionAI"
os.chdir(WORK)
sh(f"rm -rf {REPO} && git clone --depth 1 https://github.com/OmkarJayvantJadhav/MedVisionAI.git {REPO}")
os.chdir(f"{REPO}/ml")
sh("uv sync --locked")

# Always name the project's interpreter, so `uv pip` never touches the
# notebook's own Python and its preinstalled TensorFlow.
VENV_PYTHON = f"{REPO}/ml/.venv/bin/python"
tf_version = subprocess.run(
    [VENV_PYTHON, "-c", "import importlib.metadata as m; print(m.version('tensorflow'))"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f"Installing the CUDA build of TensorFlow {tf_version} (a few minutes)...")
sh(f'uv pip install --python {VENV_PYTHON} "tensorflow[and-cuda]=={tf_version}"')
# Keep the CUDA packages: stop later `uv run` calls from re-syncing them away.
os.environ["UV_NO_SYNC"] = "1"
sh("uv run python -c \"import tensorflow as tf; gpus = tf.config.list_physical_devices('GPU'); print('TensorFlow', tf.__version__, 'GPUs:', gpus); assert gpus\"")

## 3. Download ISIC 2019

In [ ]:
BASE = "https://isic-challenge-data.s3.amazonaws.com/2019"
ISIC = f"{WORK}/isic"
os.makedirs(ISIC, exist_ok=True)
for name in ["ISIC_2019_Training_GroundTruth.csv", "ISIC_2019_Training_Metadata.csv", "ISIC_2019_Training_Input.zip"]:
    print(f"Downloading {name}...")
    sh(f"cd {ISIC} && curl -sS --retry 5 -o {name} {BASE}/{name}")
print("Unzipping 25,331 images (a few minutes)...")
sh(f"cd {ISIC} && unzip -q ISIC_2019_Training_Input.zip && rm ISIC_2019_Training_Input.zip")
sh(f"ls {ISIC}/ISIC_2019_Training_Input | wc -l")

## 4. Prepare lesion-grouped splits and check for leakage

The run stops here if any lesion or duplicate image crosses splits.

In [ ]:
REPORTS = f"{OUTPUT}/reports"
os.makedirs(REPORTS, exist_ok=True)
sh(
    "uv run medvision-ml prepare-isic2019"
    f" --images {ISIC}/ISIC_2019_Training_Input"
    f" --ground-truth {ISIC}/ISIC_2019_Training_GroundTruth.csv"
    f" --metadata {ISIC}/ISIC_2019_Training_Metadata.csv"
    " --dest data/isic2019_raw"
)
print("Fingerprinting and splitting (about 10 minutes, no output until done)...")
sh("uv run medvision-ml split --source data/isic2019_raw")
print("Checking the splits for leakage (about 5 minutes)...")
sh("uv run medvision-ml check-leakage")
sh(f"cp outputs/leakage_report.json outputs/split_removed.json {REPORTS}/")
# The split holds its own copies; free about 10 GB for the image cache.
sh(f"rm -rf {ISIC} data/isic2019_raw && df -h {WORK}")

## 5. Train

Choose the experiment to run (see `docs/experiments.md`); `baseline` is the shipped recipe.
Each experiment only changes configuration settings, applied as `MEDVISION__` environment overrides.
Decoded images are cached in temporary disk space (about 15 GB) because they do not fit in RAM.
Progress is printed at the end of each epoch.

In [ ]:
EXPERIMENT = "baseline"

# Phase 9 ladder: each step adds one change to the one before (docs/experiments.md).
FULL_FINE_TUNE = {
    "TRAINING__FINE_TUNE_LAYERS": 1000,
    "TRAINING__FINE_TUNE_EPOCHS": 25,
    "TRAINING__LR_SCHEDULE": "cosine",
    "TRAINING__WARMUP_EPOCHS": 2,
}
DERMOSCOPY_AUGMENTATION = {
    "AUGMENTATION__VERTICAL_FLIP": "true",
    "AUGMENTATION__ROTATION": 0.5,
    "AUGMENTATION__BRIGHTNESS": 0.1,
    "AUGMENTATION__SATURATION": 0.1,
}
FOCAL = {"TRAINING__LOSS": "focal", "TRAINING__FOCAL_GAMMA": 2.0}
EXPERIMENTS = {
    "baseline": {},
    "full-ft": FULL_FINE_TUNE,
    "full-ft-aug": {**FULL_FINE_TUNE, **DERMOSCOPY_AUGMENTATION},
    "full-ft-aug-focal": {**FULL_FINE_TUNE, **DERMOSCOPY_AUGMENTATION, **FOCAL},
    "full-ft-aug-300": {**FULL_FINE_TUNE, **DERMOSCOPY_AUGMENTATION, "DATA__IMAGE_SIZE": 300},
    "v2s-aug-300": {
        **FULL_FINE_TUNE,
        **DERMOSCOPY_AUGMENTATION,
        "DATA__IMAGE_SIZE": 300,
        "DATA__BATCH_SIZE": 16,
        "MODEL__BACKBONE": "efficientnetv2s",
    },
}
for key, value in EXPERIMENTS[EXPERIMENT].items():
    os.environ[f"MEDVISION__{key}"] = str(value)
print(f"Experiment {EXPERIMENT}: {EXPERIMENTS[EXPERIMENT] or 'shipped recipe'}")

RUN = f"isic2019-{EXPERIMENT}"
os.environ["MEDVISION__DATA__CACHE_DIR"] = f"{WORK}/tf_cache"
sh(f"uv run medvision-ml train --run-name {RUN}")
sh(f"mkdir -p {OUTPUT}/model && cp -r artifacts/{RUN}/. {OUTPUT}/model/")
# Training history and config; the checkpoint duplicates model/.
sh(f"cp -r outputs/runs/{RUN} {REPORTS}/training && rm {REPORTS}/training/checkpoint.keras")

## 6. Evaluate on the held-out test split

In [ ]:
sh(f"uv run medvision-ml evaluate --model artifacts/{RUN}/model.keras")
sh(f"cp -r outputs/evaluation/{RUN} {REPORTS}/evaluation")
sh(f"cat outputs/evaluation/{RUN}/test/classification_report.txt")

## 7. Calibrate, fit the photo check, then export the serving artifacts

The temperature and the decision thresholds are fitted on the validation split and checked on the test split.
The out-of-distribution gate is fitted on the training and validation features and reported against 500
ordinary photos (Open Images, CC BY 2.0) downloaded here.

In [ ]:
sh(f"uv run medvision-ml calibrate --model artifacts/{RUN}/model.keras")
sh(f"cp -r outputs/calibration/{RUN} {REPORTS}/calibration")
CALIBRATION = f"outputs/calibration/{RUN}/calibration.json"

sh(f"uv run medvision-ml fetch-non-skin --dest {WORK}/non_skin --count 500")
sh(
    f"uv run medvision-ml fit-ood --model artifacts/{RUN}/model.keras"
    f" --non-skin {WORK}/non_skin --calibration {CALIBRATION}"
)
sh(f"cp -r outputs/ood/{RUN} {REPORTS}/ood")

VERSION = "skin-effb0-v2.1.0"
sh(
    f"uv run medvision-ml export --model artifacts/{RUN}/model.keras --version {VERSION}"
    f" --output export/{VERSION} --calibration {CALIBRATION} --ood outputs/ood/{RUN}/ood.json"
)
sh(f"cd export && zip -qr {OUTPUT}/{VERSION}.zip {VERSION}")
sh(f"rm -rf {WORK}/tf_cache && ls -lhR {OUTPUT}")
print(f"Done. Download {VERSION}.zip, model/ and reports/ from this version's Output tab.")